# 03 — Raw-Data EDA, Feature Engineering and Cluster Analysis
**Project:** Constraint-aware auto-ordering for school meal subscriptions (QM640).

The notebook follows the reviewer's requested sequence:
1. Load and prepare data.
2. **Raw-data EDA**: univariate analysis (§2) and bivariate analysis (§3).
3. **Round 1 clustering** on attributes *as stored*, and **feature engineering** with documented selection rules (§4).
4. **Round 2 clustering** on engineered features, choosing *k* with four criteria (§5).
5. **Comparison** of the two rounds, including ground-truth recovery (§6).
6. **Cluster EDA**: profiles and significance tests (§7).
7. **Clusters → research questions** (§8).

Heavy steps cache their results in `cache/`, so re-running is fast. Delete `cache/` to recompute from scratch. Figures are saved in `fig/`.

In [ ]:
# ---------------- configuration ----------------
DATA_DIR = "../data/raw"     # folder with the CSV files
CACHE = "cache"; FIG = "fig"           # intermediate results / figures
TRAIN_END = "2025-03-31"               # features & clusters use data up to here only
TEST_START = "2025-04-01"
MIN_ORDERS = 10                        # minimum live training orders for a student to be clustered
SEED = 42
import os, json, math, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")
os.makedirs(CACHE, exist_ok=True); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 9, "savefig.dpi": 160})
NAVY, ORANGE = "#1f3b63", "#c97b2a"
R_PATH = os.path.join(CACHE, "results.json")
R = json.load(open(R_PATH)) if os.path.exists(R_PATH) else {}
def save_R(): json.dump(R, open(R_PATH, "w"), indent=1, default=float)
def savefig(name): plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png"); plt.show()
def cache(name, fn):
    p = os.path.join(CACHE, name + ".pkl")
    if os.path.exists(p): return pd.read_pickle(p)
    obj = fn(); pd.to_pickle(obj, p); return obj
def cramers_v(ct):
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum(); r, k = ct.shape
    return chi2, p, dof, math.sqrt(chi2 / (n * (min(r, k) - 1)))
def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n
def describe_num(s):
    s = pd.Series(s).dropna()
    return {"n": int(len(s)), "mean": float(s.mean()), "sd": float(s.std()), "median": float(s.median()),
            "q1": float(s.quantile(.25)), "q3": float(s.quantile(.75)), "min": float(s.min()), "max": float(s.max()),
            "skew": float(stats.skew(s)), "kurtosis": float(stats.kurtosis(s))}
BAND = {"PS": "K-2", "TK": "K-2", "K": "K-2", "1": "K-2", "2": "K-2", "3": "3-5", "4": "3-5", "5": "3-5",
        "6": "6-8", "7": "6-8", "8": "6-8", "9": "9-12", "10": "9-12", "11": "9-12", "12": "9-12", "Staff": "Staff"}
GNUM = {"PS": -1, "TK": -0.5, "K": 0, **{str(i): i for i in range(1, 13)}, "Staff": 14, "Unknown": np.nan}
BANDS = ["K-2", "3-5", "6-8", "9-12", "Staff"]

## 1. Load and prepare
Live lunches are those with an empty `LUNCHCANCELED`. IOU emergency lunches are excluded from behavioural analysis. Each lunch is linked to school, age band and packing group, and the number of live sides is attached.

In [ ]:
# ---------------- 1. load & prepare ----------------
def _load(name, **kw):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"), encoding="utf-8-sig", **kw)
    obj = df.select_dtypes("object").columns
    df[obj] = df[obj].replace('""', np.nan)
    return df
def _prep():
    L = _load("STUDENTLUNCH", usecols=["STUDENTLUNCHID", "STUDENTID", "GRADEID", "MENUITEMID", "PORTIONCODE", "DELIVERYDATE",
                                       "STUDENTTYPE", "LUNCHTYPE", "ORDERCHANNEL", "LUNCHCREATED", "LUNCHCANCELED", "ORDERCHANNELDETAIL"],
              dtype={"STUDENTID": "int32", "GRADEID": "int32", "MENUITEMID": "int16", "PORTIONCODE": "int8",
                     "STUDENTTYPE": "category", "LUNCHTYPE": "category", "ORDERCHANNEL": "category", "ORDERCHANNELDETAIL": "category"},
              parse_dates=["DELIVERYDATE", "LUNCHCREATED", "LUNCHCANCELED"])
    A = _load("STUDENTLUNCHADDON", usecols=["STUDENTLUNCHID", "COMPONENTID", "ADDONCANCELED"])
    G = _load("GRADE"); SG = _load("SERVINGGROUP"); SC = _load("SCHOOL"); M = _load("MENUITEM"); CU = _load("CUISINE")
    g = G.merge(SG[["SERVINGGROUPID", "PACKINGGROUPID"]], on="SERVINGGROUPID", how="left")
    g = g.merge(SC[["SCHOOLID", "TYPE"]].rename(columns={"TYPE": "SCHOOLTYPE"}), on="SCHOOLID", how="left")
    g["BAND"] = g.GRADE.astype(str).map(BAND); g["GNUM"] = g.GRADE.astype(str).map(GNUM)
    L = L.merge(g[["GRADEID", "SCHOOLID", "PACKINGGROUPID", "GRADE", "BAND", "GNUM", "SCHOOLTYPE"]], on="GRADEID", how="left")
    M = M.merge(CU, on="CUISINEID", how="left")
    L = L.merge(M[["MENUITEMID", "ITEMNAME", "CUISINENAME", "ADVENTUROUSRATING", "SERVINGTYPE"]], on="MENUITEMID", how="left")
    sides = A[A.ADDONCANCELED.isna()].groupby("STUDENTLUNCHID").size()
    L["SIDES"] = L.STUDENTLUNCHID.map(sides).fillna(0).astype("int8")
    L["CANCELLED"] = L.LUNCHCANCELED.notna()
    L["IS_SUB"] = L.STUDENTTYPE.astype(str).str.startswith("Sub")
    L["LEAD_H"] = (L.DELIVERYDATE + pd.Timedelta(hours=11) - L.LUNCHCREATED).dt.total_seconds() / 3600
    L["WD"] = L.DELIVERYDATE.dt.dayofweek.astype("int8")
    for c in ["ITEMNAME", "CUISINENAME", "GRADE", "BAND", "SCHOOLTYPE", "SERVINGTYPE"]:
        L[c] = L[c].astype("category")                      # text -> category: ~4x less memory
    L["LEAD_H"] = L.LEAD_H.astype("float32"); L["GNUM"] = L.GNUM.astype("float32")
    L["SCHOOLID"] = L.SCHOOLID.astype("int32"); L["PACKINGGROUPID"] = L.PACKINGGROUPID.astype("int32")
    return L.drop(columns=["ORDERCHANNELDETAIL"])
lunch = cache("lunch", _prep)
student = cache("student", lambda: _load("STUDENT", parse_dates=["DATEREGISTERED"]))
stu_alg = cache("stu_alg", lambda: _load("STUDENTALLERGENS"))
item = cache("item", lambda: _load("MENUITEM").merge(_load("CUISINE"), on="CUISINEID", how="left"))
menu = cache("menu", lambda: _load("TBLPACKINGGROUPMENUS", usecols=["SCHOOLID", "PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID", "PORTIONCODE"],
                                   parse_dates=["DELIVERYDATE"], dtype={"MENUITEMID": "int16", "PORTIONCODE": "int8"}))
live = lunch[~lunch.CANCELLED & (lunch.LUNCHTYPE != "IOU")]
train = live[live.DELIVERYDATE <= TRAIN_END]; test = live[live.DELIVERYDATE >= TEST_START]
R["sizes"] = {"lunch_rows": len(lunch), "live": len(live), "train_live": len(train), "test_live": len(test),
              "students": len(student)}
save_R(); print(R["sizes"])

## 2. Raw-data EDA — univariate
Each variable is examined on its own: shares for categorical variables, and centre, spread, skewness and kurtosis for numeric ones. Two concentration measures are used:
* **Gini coefficient** of item popularity: 0 means every item is ordered equally, 1 means one item takes everything.
* **Top-10 share:** the proportion of all lunches taken by the ten most popular items.

In [ ]:
# ---------------- 2. RAW DATA EDA: UNIVARIATE ----------------
U = {}
# 2a categorical lunch-level variables
U["lunchtype"] = lunch.LUNCHTYPE.value_counts(normalize=True).round(4).to_dict()
U["channel"] = lunch.ORDERCHANNEL.astype(str).replace("nan", "(none: auto/IOU)").value_counts(normalize=True).round(4).to_dict()
U["portion"] = live.PORTIONCODE.map({1: "Regular", 2: "Large"}).value_counts(normalize=True).round(4).to_dict()
U["cancel_rate"] = float(lunch[lunch.LUNCHTYPE != "IOU"].CANCELLED.mean())
U["sides_dist"] = live.SIDES.clip(upper=3).value_counts(normalize=True).sort_index().round(4).to_dict()
U["weekday"] = live.WD.map(dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri"]))).value_counts(normalize=True).round(4).to_dict()
# 2b numeric lunch-level: lead time, daily volume, sides
lead = lunch.loc[lunch.LUNCHTYPE.isin(["PreOrder", "LastMinute"]), "LEAD_H"]
U["lead_time_h"] = describe_num(lead)
daily = live.groupby("DELIVERYDATE").size()
U["daily_volume"] = describe_num(daily)
U["sides_per_lunch"] = describe_num(live.SIDES)
# 2c item popularity (concentration)
pop = live.ITEMNAME.value_counts()
U["items_ordered"] = int(pop.size); U["item_gini"] = float(gini(pop.values))
U["top10_item_share"] = float(pop.head(10).sum() / pop.sum()); U["top5_items"] = pop.head(5).round(0).to_dict()
U["cuisine"] = live.CUISINENAME.value_counts(normalize=True).round(4).to_dict()
# 2d student-level raw fields
st = student.copy()
gsel = lunch.drop_duplicates("STUDENTID", keep="last").set_index("STUDENTID")[["GRADE", "BAND", "GNUM", "SCHOOLTYPE"]]
st = st.join(gsel, on="STUDENTID")
st["ALG_N"] = st.STUDENTID.map(stu_alg[stu_alg.ACTIVECODE.astype(str).str.lower() == "true"].groupby("STUDENTID").size()).fillna(0)
st["N_LUNCH"] = st.STUDENTID.map(live.groupby("STUDENTID").size()).fillna(0)
st["REG_YEAR"] = st.DATEREGISTERED.dt.year
st["TYPE_GRP"] = np.where(st.STUDENTTYPE.astype(str).str.startswith("Sub"), "Subscription", st.STUDENTTYPE.astype(str))
U["student_type"] = st.TYPE_GRP.value_counts(normalize=True).round(4).to_dict()
U["band"] = st.BAND.value_counts(normalize=True).round(4).to_dict()
U["palette"] = st.PALETTERATING.value_counts(normalize=True).sort_index().round(4).to_dict()
U["default_portion"] = st.DEFAULTPORTIONCODE.value_counts(normalize=True).sort_index().round(4).to_dict()
U["allergen_n"] = st.ALG_N.clip(upper=3).value_counts(normalize=True).sort_index().round(4).to_dict()
U["active"] = float(st.ACTIVECODE.astype(str).str.lower().eq("true").mean())
U["lunches_per_student"] = describe_num(st.N_LUNCH); U["lunch_gini_students"] = float(gini(st.N_LUNCH.values))
U["never_ordered"] = float((st.N_LUNCH == 0).mean())
U["missing_pct"] = (lunch.isna().mean() * 100).round(2)[lambda s: s > 0].to_dict()
R["univariate"] = U; save_R()
pd.to_pickle(st, os.path.join(CACHE, "student_raw.pkl"))

# figures
fig, ax = plt.subplots(2, 3, figsize=(11, 6))
pd.Series(U["lunchtype"]).plot.bar(ax=ax[0, 0], color=NAVY, title="(a) Lunch type share")
ax[0, 1].hist(np.log10(lead.clip(lower=0.5)), bins=60, color=NAVY); ax[0, 1].set_title("(b) Order lead time (log10 hours)")
ax[0, 2].hist(daily.values, bins=50, color=NAVY); ax[0, 2].set_title("(c) Live lunches per delivery day")
ax[1, 0].plot(np.arange(1, pop.size + 1), pop.values, color=NAVY); ax[1, 0].set_yscale("log")
ax[1, 0].set_title(f"(d) Item rank-frequency (Gini {U['item_gini']:.2f})"); ax[1, 0].set_xlabel("item rank")
pd.Series(U["sides_dist"]).plot.bar(ax=ax[1, 1], color=NAVY, title="(e) Sides per live lunch (3 = 3+)")
ax[1, 2].hist(st.N_LUNCH.clip(upper=300), bins=60, color=NAVY); ax[1, 2].set_title("(f) Live lunches per student (clipped 300)")
savefig("u01_lunch_univariate")
fig, ax = plt.subplots(1, 4, figsize=(11, 2.8))
st.BAND.value_counts().reindex(BANDS).plot.bar(ax=ax[0], color=NAVY, title="Age band")
st.TYPE_GRP.value_counts().plot.bar(ax=ax[1], color=NAVY, title="Student type")
st.PALETTERATING.value_counts().sort_index().plot.bar(ax=ax[2], color=NAVY, title="Palette rating")
st.ALG_N.clip(upper=3).value_counts().sort_index().plot.bar(ax=ax[3], color=NAVY, title="Active allergens (3=3+)")
savefig("u02_student_univariate")
print(json.dumps({k: U[k] for k in ["lead_time_h", "daily_volume", "item_gini", "top10_item_share", "lunches_per_student", "lunch_gini_students", "never_ordered", "cancel_rate"]}, indent=0, default=float))

## 3. Raw-data EDA — bivariate
Pairs of variables are tested with a method matched to their types:
* categorical × categorical → **chi-square test** with **Cramér's V** as the effect size
* numeric × binary → **Mann–Whitney U** with **rank-biserial correlation** (the numbers are skewed, so a rank-based test is safer than a t test)
* ordinal × numeric → **Spearman's ρ**
* one categorical variable against equal shares → **chi-square goodness-of-fit**

These results feed directly into the feature decisions in §4.

In [ ]:
# ---------------- 3. RAW DATA EDA: BIVARIATE ----------------
st = pd.read_pickle(os.path.join(CACHE, "student_raw.pkl"))
BV = {}
nonIOU = lunch[lunch.LUNCHTYPE != "IOU"]
# (a) lunch type x cancellation  -> chi-square + Cramer's V
ct = pd.crosstab(nonIOU.LUNCHTYPE.astype(str), nonIOU.CANCELLED)
chi2, p, dof, v = cramers_v(ct)
BV["lunchtype_x_cancel"] = {"rates": nonIOU.groupby(nonIOU.LUNCHTYPE.astype(str)).CANCELLED.mean().round(4).to_dict(),
                            "chi2": chi2, "dof": dof, "p": p, "cramers_v": v}
# (b) lead time x cancellation (manual orders) -> Mann-Whitney + rank-biserial; binned rates
man = nonIOU[nonIOU.LUNCHTYPE.isin(["PreOrder", "LastMinute"])].dropna(subset=["LEAD_H"])
a, b = man.loc[man.CANCELLED, "LEAD_H"], man.loc[~man.CANCELLED, "LEAD_H"]
samp_a, samp_b = a.sample(min(len(a), 50000), random_state=SEED), b.sample(min(len(b), 50000), random_state=SEED)
U_, p = stats.mannwhitneyu(samp_a, samp_b)
BV["lead_x_cancel"] = {"median_cancelled": float(a.median()), "median_kept": float(b.median()), "U": float(U_), "p": p,
                       "rank_biserial": float(1 - 2 * U_ / (len(samp_a) * len(samp_b)))}
bins = pd.cut(man.LEAD_H, [0, 12, 24, 72, 168, 336, 720, 2000], labels=["<12h", "12-24h", "1-3d", "3-7d", "1-2w", "2-4w", ">4w"])
BV["lead_bin_cancel"] = man.groupby(bins).CANCELLED.mean().round(4).to_dict()
# (c) age band x cuisine -> Cramer's V
ct = pd.crosstab(live.BAND, live.CUISINENAME); chi2, p, dof, v = cramers_v(ct)
BV["band_x_cuisine"] = {"chi2": chi2, "dof": dof, "p": p, "cramers_v": v}
# (d) age band x portion
ct = pd.crosstab(live.BAND, live.PORTIONCODE); chi2, p, dof, v = cramers_v(ct)
BV["band_x_portion"] = {"large_share": live.groupby("BAND").PORTIONCODE.apply(lambda x: (x == 2).mean()).round(4).to_dict(),
                        "chi2": chi2, "p": p, "cramers_v": v}
# (e) weekday x volume -> chi-square goodness of fit vs uniform
obs = live.WD.value_counts().sort_index().values
chi2, p = stats.chisquare(obs)
BV["weekday_uniform"] = {"chi2": chi2, "p": p, "counts": obs.tolist()}
# (f) subscription x sides -> Mann-Whitney
s1, s0 = live.loc[live.IS_SUB, "SIDES"], live.loc[~live.IS_SUB, "SIDES"]
U_, p = stats.mannwhitneyu(s1.sample(50000, random_state=SEED), s0.sample(50000, random_state=SEED))
BV["sub_x_sides"] = {"mean_sub": float(s1.mean()), "mean_nonsub": float(s0.mean()), "p": p,
                     "rank_biserial": float(1 - 2 * U_ / 50000 ** 2)}
# (g) palette rating (raw field) x adventurousness actually chosen -> Spearman
adv = train.groupby("STUDENTID").ADVENTUROUSRATING.mean()
m = st.set_index("STUDENTID").join(adv.rename("ADV_CHOSEN"), how="inner")
rho, p = stats.spearmanr(m.PALETTERATING, m.ADV_CHOSEN)
BV["palette_x_adv"] = {"spearman_rho": rho, "p": p, "n": int(len(m))}
# (h) grade (numeric) x adventurousness chosen
m2 = m.dropna(subset=["GNUM"]); m2 = m2[m2.GNUM < 14]
rho, p = stats.spearmanr(m2.GNUM, m2.ADV_CHOSEN); BV["grade_x_adv"] = {"spearman_rho": rho, "p": p}
# (i) correlation matrix of student-level raw numeric fields
num = st[["GNUM", "PALETTERATING", "DEFAULTPORTIONCODE", "ALG_N", "REG_YEAR", "N_LUNCH"]].copy()
num["IS_SUB"] = (st.TYPE_GRP == "Subscription").astype(int)
corr = num.corr(method="spearman"); BV["raw_corr"] = corr.round(3).to_dict()
# (j) month x volume (seasonality)
BV["month_volume"] = live.groupby(live.DELIVERYDATE.dt.to_period("M")).size().astype(int).rename(str).to_dict()
R["bivariate"] = BV; save_R()

fig, ax = plt.subplots(2, 3, figsize=(11, 6.2))
pd.Series(BV["lunchtype_x_cancel"]["rates"]).plot.bar(ax=ax[0, 0], color=NAVY, title="(a) Cancellation rate by lunch type")
pd.Series(BV["lead_bin_cancel"]).plot.bar(ax=ax[0, 1], color=NAVY, title="(b) Cancellation by order lead time")
cz = pd.crosstab(live.BAND, live.CUISINENAME, normalize="index").reindex(BANDS)
im = ax[0, 2].imshow(cz.values, cmap="Blues", aspect="auto"); ax[0, 2].set_yticks(range(5), BANDS)
ax[0, 2].set_xticks(range(cz.shape[1]), [c[:9] for c in cz.columns], rotation=70, fontsize=7); ax[0, 2].set_title("(c) Cuisine share by age band"); ax[0, 2].grid(False)
pd.Series(BV["band_x_portion"]["large_share"]).reindex(BANDS).plot.bar(ax=ax[1, 0], color=NAVY, title="(d) Large-portion share by band")
sm = m.sample(4000, random_state=SEED)
ax[1, 1].boxplot([m.loc[m.PALETTERATING == r, "ADV_CHOSEN"] for r in range(4)], labels=["0", "1", "2", "3"])
ax[1, 1].set_title(f"(e) Adventurousness chosen by palette rating (ρ={BV['palette_x_adv']['spearman_rho']:.2f})")
im2 = ax[1, 2].imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1); ax[1, 2].set_xticks(range(len(corr)), corr.columns, rotation=60, fontsize=7)
ax[1, 2].set_yticks(range(len(corr)), corr.columns, fontsize=7); ax[1, 2].set_title("(f) Spearman correlation, raw student fields"); ax[1, 2].grid(False)
for i in range(len(corr)):
    for j in range(len(corr)): ax[1, 2].text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
savefig("b01_bivariate")
mv = pd.Series(BV["month_volume"]); plt.figure(figsize=(9, 2.8)); plt.bar(range(len(mv)), mv.values / 1000, color=NAVY)
plt.xticks(range(len(mv)), mv.index, rotation=70, fontsize=7); plt.ylabel("thousands"); plt.title("Live lunches by month"); savefig("b02_month_volume")
print(json.dumps({k: v for k, v in BV.items() if k not in ("raw_corr", "month_volume")}, indent=0, default=float)[:2500])

## 4. Features — Round 1 (as stored) and Round 2 (engineered)
* **Population:** students with at least 10 live lunches in the training window. Below that, shares such as favourite share are too noisy: their standard error is up to 0.16.
* **Round 1:** the student attributes exactly as stored (grade, palette rating, default portion, allergen count, registration year, active flag, student type, school type). Categories are one-hot encoded; nothing behavioural is derived.
* **Round 2:** 15 candidate behavioural features computed from training data only. They are screened by:
  1. **Redundancy:** |Spearman ρ| > 0.75, meaning the pair shares more than 56% of its rank variance.
  2. **Relevance and confounding:** evidence from §3.
  3. **Multicollinearity:** variance inflation factor (VIF), which must stay below 5.
  4. **Transformation:** Yeo–Johnson where |skewness| > 1, then z-scoring, so that k-means's Euclidean distance weighs every feature equally.

In [ ]:
# ---------------- 4. FEATURES: Round 1 (as stored) and Round 2 (engineered) ----------------
st = pd.read_pickle(os.path.join(CACHE, "student_raw.pkl")).set_index("STUDENTID")
n_tr = train.groupby("STUDENTID").size()
elig = n_tr[n_tr >= MIN_ORDERS].index
R["eligible_students"] = int(len(elig))

# ---- Round 1: attributes exactly as stored (no behavioural derivation) ----
r1 = st.loc[elig, ["GNUM", "PALETTERATING", "DEFAULTPORTIONCODE", "ALG_N", "REG_YEAR"]].copy()
r1["ACTIVE"] = st.loc[elig, "ACTIVECODE"].astype(str).str.lower().eq("true").astype(int)
r1 = r1.join(pd.get_dummies(st.loc[elig, "TYPE_GRP"], prefix="TYPE").astype(int))
r1 = r1.join(pd.get_dummies(st.loc[elig, "SCHOOLTYPE"].fillna("Unknown"), prefix="SCH").astype(int))
r1["GNUM"] = r1.GNUM.fillna(r1.GNUM.median())
pd.to_pickle(r1, os.path.join(CACHE, "r1.pkl"))

# ---- Round 2: engineered behavioural features (training window only) ----
t = train[train.STUDENTID.isin(elig)].sort_values(["STUDENTID", "DELIVERYDATE"])
g = t.groupby("STUDENTID")
f = pd.DataFrame(index=elig)
span_w = ((g.DELIVERYDATE.max() - g.DELIVERYDATE.min()).dt.days / 7).clip(lower=1)
f["orders_per_week"] = g.size() / span_w
cnt = t.groupby(["STUDENTID", "MENUITEMID"]).size()
f["variety_ratio"] = cnt.groupby(level=0).size() / g.size()
f["fav_share"] = cnt.groupby(level=0).max() / g.size()
p_ = cnt / cnt.groupby(level=0).transform("sum")
H = -(p_ * np.log(p_)).groupby(level=0).sum()
f["item_entropy"] = H / np.log(cnt.groupby(level=0).size().clip(lower=2))
f["repeat_rate"] = t.MENUITEMID.eq(g.MENUITEMID.shift()).groupby(t.STUDENTID).mean()
f["adv_mean"] = g.ADVENTUROUSRATING.mean()
COMFORT = {"Classic American", "Italian", "Breakfast for Lunch"}
f["comfort_share"] = t.CUISINENAME.isin(COMFORT).groupby(t.STUDENTID).mean()
cc = t.groupby(["STUDENTID", "CUISINENAME"]).size(); pc = cc / cc.groupby(level=0).transform("sum")
f["cuisine_entropy"] = -(pc * np.log(pc)).groupby(level=0).sum() / np.log(10)
f["sides_mean"] = g.SIDES.mean()
wd = t.groupby(["STUDENTID", "WD"]).size(); f["weekday_conc"] = wd.groupby(level=0).max() / g.size()
f["sub_share"] = g.IS_SUB.mean()
f["large_share"] = (t.PORTIONCODE == 2).groupby(t.STUDENTID).mean()
f["cold_share"] = (t.SERVINGTYPE == "Cold").groupby(t.STUDENTID).mean()
placed = lunch[(lunch.DELIVERYDATE <= TRAIN_END) & (lunch.LUNCHTYPE != "IOU") & lunch.STUDENTID.isin(elig)]
f["cancel_rate"] = placed.groupby("STUDENTID").CANCELLED.mean()
f["lead_time_med"] = t[t.LUNCHTYPE.isin(["PreOrder", "LastMinute"])].groupby("STUDENTID").LEAD_H.median()
f = f.fillna(f.median())
CANDIDATES = list(f.columns)

# ---- feature screening ----
FS = {}
FS["describe"] = {c: describe_num(f[c]) for c in CANDIDATES}
sp = f.corr(method="spearman"); FS["spearman"] = sp.round(3).to_dict()
# drop rule 1 (redundancy): |rho| > 0.75 means the pair shares > 56% of rank variance (rho^2);
# keep the member that is easier to interpret and bounded in [0, 1]
CORR_T = 0.75
PREFER = ["fav_share", "variety_ratio", "orders_per_week", "adv_mean", "comfort_share", "cuisine_entropy"]
pairs = [(a, b, sp.loc[a, b]) for i, a in enumerate(CANDIDATES) for b in CANDIDATES[i + 1:] if abs(sp.loc[a, b]) > CORR_T]
FS["high_corr_pairs"] = [(a, b, round(float(r), 3)) for a, b, r in pairs]
DROP = {}
for a, b, r in pairs:
    keep = a if a in PREFER else b; drop = b if keep == a else a
    DROP[drop] = f"redundant with {keep} (Spearman rho = {r:.2f}, |rho| > {CORR_T})"
# drop rule 2 (relevance / structural confounding), evidence from the bivariate EDA
DROP.setdefault("lead_time_med", "no association with cancellation (rank-biserial -0.007) and |rho| < 0.07 with every other "
                "feature: reflects calendar habits, not food preference")
DROP.setdefault("large_share", "structural: set by grade and portion availability (Cramer's V 0.48 with age band; Large not "
                "offered to K-2), so it would cluster on age rather than behaviour")
SELECTED = [c for c in CANDIDATES if c not in DROP]
# VIF on selected (after standardising)
from statsmodels.stats.outliers_influence import variance_inflation_factor
Z = (f[SELECTED] - f[SELECTED].mean()) / f[SELECTED].std()
FS["vif"] = {c: float(variance_inflation_factor(Z.values, i)) for i, c in enumerate(SELECTED)}
# transformation: Yeo-Johnson for |skew| > 1, then z-score
from sklearn.preprocessing import PowerTransformer, StandardScaler
skewed = [c for c in SELECTED if abs(stats.skew(f[c])) > 1]
X2 = f[SELECTED].copy()
if skewed: X2[skewed] = PowerTransformer(method="yeo-johnson").fit_transform(X2[skewed])
X2 = pd.DataFrame(StandardScaler().fit_transform(X2), index=f.index, columns=SELECTED)
FS.update({"candidates": CANDIDATES, "dropped": DROP, "selected": SELECTED, "yeo_johnson": skewed,
           "skew_after": {c: float(stats.skew(X2[c])) for c in SELECTED}})
R["features"] = FS; save_R()
pd.to_pickle(f, os.path.join(CACHE, "f_raw.pkl")); pd.to_pickle(X2, os.path.join(CACHE, "X2.pkl"))

fig, axes = plt.subplots(3, 6, figsize=(13, 6))
for ax, c in zip(axes.flat, CANDIDATES):
    ax.hist(f[c], bins=40, color=NAVY if c in SELECTED else "#aaaaaa"); ax.set_title(c + ("" if c in SELECTED else " (dropped)"), fontsize=8)
for ax in axes.flat[len(CANDIDATES):]: ax.axis("off")
savefig("f01_feature_distributions")
fig, ax = plt.subplots(figsize=(7.5, 6.2)); im = ax.imshow(sp.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sp)), sp.columns, rotation=75, fontsize=7); ax.set_yticks(range(len(sp)), sp.columns, fontsize=7)
for i in range(len(sp)):
    for j in range(len(sp)): ax.text(j, i, f"{sp.values[i, j]:.2f}", ha="center", va="center", fontsize=5.5)
ax.grid(False); ax.set_title("Spearman correlation of candidate features"); plt.colorbar(im); savefig("f02_feature_correlation")
print("eligible", len(elig)); print("pairs", FS["high_corr_pairs"]); print("vif", {k: round(v, 2) for k, v in FS["vif"].items()}); print("skewed", skewed)

## 5. Choosing the number of clusters
K-means is fitted for k = 2 … 10, and each solution is scored on:
* **Elbow:** within-cluster sum of squares.
* **Silhouette:** higher is better.
* **Davies–Bouldin:** lower is better.
* **Calinski–Harabasz:** higher is better.
* **Stability:** mean adjusted Rand index (ARI) between the full fit and fits on 80% subsamples.

A Gaussian mixture model's BIC is a model-based cross-check.

In [ ]:
# ---------------- 5. CLUSTERING: choose k for Round 1 and Round 2 ----------------
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
r1 = pd.read_pickle(os.path.join(CACHE, "r1.pkl"))
X1 = pd.DataFrame(StandardScaler().fit_transform(r1), index=r1.index, columns=r1.columns)
X2 = pd.read_pickle(os.path.join(CACHE, "X2.pkl"))
pd.to_pickle(X1, os.path.join(CACHE, "X1.pkl"))
rng = np.random.default_rng(SEED)
def scan(X, label, ks=range(2, 11)):
    Xv = X.values; idx = rng.choice(len(Xv), size=min(6000, len(Xv)), replace=False)
    rows = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xv)
        lab = km.labels_
        # stability: re-fit on 5 random 80% subsamples, predict all, ARI vs full fit
        aris = []
        for s in range(5):
            sub = rng.choice(len(Xv), size=int(.8 * len(Xv)), replace=False)
            km_s = KMeans(n_clusters=k, n_init=5, random_state=100 + s).fit(Xv[sub])
            aris.append(adjusted_rand_score(lab, km_s.predict(Xv)))
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Xv[idx], lab[idx]),
                     "davies_bouldin": davies_bouldin_score(Xv, lab), "calinski_harabasz": calinski_harabasz_score(Xv, lab),
                     "stability_ari": float(np.mean(aris)), "min_cluster_share": float(np.bincount(lab).min() / len(lab))})
        print(label, k, {k_: round(v, 3) for k_, v in rows[-1].items() if k_ != "k"}, flush=True)
    return pd.DataFrame(rows)
S1 = cache("scan_r1", lambda: scan(X1, "R1"))
S2 = cache("scan_r2", lambda: scan(X2, "R2"))
# GMM BIC cross-check (Round 2)
def gmm_bic():
    idx = rng.choice(len(X2), size=8000, replace=False)
    return pd.DataFrame([{"k": k, "bic": GaussianMixture(k, covariance_type="full", random_state=SEED).fit(X2.values[idx]).bic(X2.values[idx])}
                         for k in range(2, 11)])
B2 = cache("gmm_bic_r2", gmm_bic)
R["scan_r1"] = S1.round(4).to_dict("records"); R["scan_r2"] = S2.round(4).to_dict("records"); R["gmm_bic_r2"] = B2.round(1).to_dict("records")
save_R()
fig, ax = plt.subplots(1, 4, figsize=(13, 3))
for S, lab, c in [(S1, "Round 1 (as stored)", ORANGE), (S2, "Round 2 (engineered)", NAVY)]:
    ax[0].plot(S.k, S.inertia / S.inertia.iloc[0], marker="o", color=c, label=lab)
    ax[1].plot(S.k, S.silhouette, marker="o", color=c); ax[2].plot(S.k, S.davies_bouldin, marker="o", color=c)
    ax[3].plot(S.k, S.stability_ari, marker="o", color=c)
ax[0].set_title("Elbow: inertia (relative to k=2)"); ax[1].set_title("Silhouette (higher better)")
ax[2].set_title("Davies-Bouldin (lower better)"); ax[3].set_title("Stability ARI (higher better)"); ax[0].legend(fontsize=7)
for a in ax: a.set_xlabel("k")
savefig("c01_k_selection")
plt.figure(figsize=(4.5, 3)); plt.plot(B2.k, B2.bic, marker="o", color=NAVY); plt.title("GMM BIC, Round 2 (lower better)"); plt.xlabel("k"); savefig("c02_gmm_bic")

## 6. Final clusters and Round 1 vs Round 2
Round 2 uses **k = 3**: the best Davies–Bouldin and Calinski–Harabasz scores, stability ARI of 0.99, and the BIC elbow. Round 1 is compared at the same k, and also at its own best-silhouette k.

**Ground-truth recovery** uses η² (eta squared): the share of each hidden trait's variance that lies *between* clusters. The hidden traits are used **only** for this check, never as features.

In [ ]:
# ---------------- 6. FINAL CLUSTERS + ROUND 1 vs ROUND 2 ----------------
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.decomposition import PCA
X1 = pd.read_pickle(os.path.join(CACHE, "X1.pkl")); X2 = pd.read_pickle(os.path.join(CACHE, "X2.pkl"))
K2 = 3          # chosen: best Davies-Bouldin & Calinski-Harabasz, stability ARI 0.99, BIC elbow
S1 = pd.DataFrame(R["scan_r1"]); K1_BEST = int(S1.loc[S1.silhouette.idxmax(), "k"])
km2 = KMeans(K2, n_init=20, random_state=SEED).fit(X2.values)
km1 = KMeans(K2, n_init=20, random_state=SEED).fit(X1.values)
km1b = KMeans(K1_BEST, n_init=20, random_state=SEED).fit(X1.values)
lab2, lab1, lab1b = km2.labels_, km1.labels_, km1b.labels_
rng = np.random.default_rng(SEED); idx = rng.choice(len(X2), 6000, replace=False)
def quality(X, lab):
    return {"silhouette": float(silhouette_score(X.values[idx], lab[idx])), "davies_bouldin": float(davies_bouldin_score(X.values, lab)),
            "calinski_harabasz": float(calinski_harabasz_score(X.values, lab)), "sizes": np.bincount(lab).tolist()}
CMP = {"R1_k3": quality(X1, lab1), f"R1_k{K1_BEST}": quality(X1, lab1b), "R2_k3": quality(X2, lab2)}
# algorithm cross-checks for Round 2
gm = GaussianMixture(K2, covariance_type="full", random_state=SEED).fit(X2.values)
CMP["R2_ari_kmeans_vs_gmm"] = float(adjusted_rand_score(lab2, gm.predict(X2.values)))
wi = rng.choice(len(X2), 5000, replace=False)
ward = AgglomerativeClustering(K2, linkage="ward").fit_predict(X2.values[wi])
CMP["R2_ari_kmeans_vs_ward"] = float(adjusted_rand_score(lab2[wi], ward))
# what drives Round 1 clusters? share of each dummy per cluster
r1 = pd.read_pickle(os.path.join(CACHE, "r1.pkl"))
CMP["R1_k3_profile"] = r1.groupby(lab1).mean().round(3).to_dict("index")
# ground-truth recovery: eta^2 = SS_between / SS_total of hidden traits
gt = pd.read_csv(os.path.join(DATA_DIR, "_ground_truth", "STUDENT_LATENT_TRAITS.csv")).set_index("STUDENTID").loc[X2.index]
def eta2(y, lab):
    y = np.asarray(y, float); m = y.mean()
    return float(sum(((y[lab == c].mean() - m) ** 2) * (lab == c).sum() for c in np.unique(lab)) / ((y - m) ** 2).sum())
TRAITS = ["loyalty", "adventurousness", "nonsub_weekly_freq", "side_rate", "fatigue", "novelty_seeking"]
CMP["eta2"] = {t: {"R1_k3": eta2(gt[t], lab1), f"R1_k{K1_BEST}": eta2(gt[t], lab1b), "R2_k3": eta2(gt[t], lab2)} for t in TRAITS}
R["compare"] = CMP; R["K2"] = K2; R["K1_BEST"] = K1_BEST; save_R()
labels = pd.DataFrame({"R1_k3": lab1, f"R1_k{K1_BEST}": lab1b, "R2": lab2}, index=X2.index)
pd.to_pickle(labels, os.path.join(CACHE, "labels.pkl")); pd.to_pickle(km2, os.path.join(CACHE, "km2.pkl"))
# PCA views
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, X, lab, t in [(ax[0], X1, lab1, "Round 1: as-stored attributes (k=3)"), (ax[1], X2, lab2, "Round 2: engineered features (k=3)")]:
    P = PCA(2, random_state=SEED).fit(X.values); Z = P.transform(X.values[idx])
    a.scatter(Z[:, 0], Z[:, 1], c=lab[idx], s=3, cmap="tab10", alpha=.6)
    a.set_title(f"{t}\nPC1+PC2 explain {P.explained_variance_ratio_[:2].sum():.0%}"); a.set_xlabel("PC1"); a.set_ylabel("PC2")
savefig("c03_pca_r1_vs_r2")
e = pd.DataFrame(CMP["eta2"]).T
e.plot.bar(figsize=(8, 3.2), color=["#e6b98a", ORANGE, NAVY], title="Ground-truth recovery: share of hidden-trait variance explained by clusters (η²)")
plt.xticks(rotation=20); savefig("c04_ground_truth_eta2")
print(json.dumps({k: v for k, v in CMP.items() if k != "R1_k3_profile"}, indent=0, default=float)[:3000])
print(pd.DataFrame(CMP["R1_k3_profile"]).T.round(2).to_string())

## 7. Cluster EDA
Each cluster is profiled by its mean and z-score on every engineered feature. Differences between clusters are tested with:
* **Kruskal–Wallis H** and its effect size ε² for numeric features
* **chi-square** and **Cramér's V** for age band, school type and plan days

In [ ]:
# ---------------- 7. CLUSTER EDA: profiles and tests ----------------
labels = pd.read_pickle(os.path.join(CACHE, "labels.pkl")); f = pd.read_pickle(os.path.join(CACHE, "f_raw.pkl"))
st = pd.read_pickle(os.path.join(CACHE, "student_raw.pkl")).set_index("STUDENTID")
SEL = R["features"]["selected"]
lab = labels.R2
prof = f.loc[labels.index, SEL].groupby(lab).mean()
zprof = (prof - f[SEL].mean()) / f[SEL].std()
P = {"profile_mean": prof.round(4).to_dict("index"), "profile_z": zprof.round(3).to_dict("index"),
     "sizes": lab.value_counts().sort_index().to_dict()}
# Kruskal-Wallis per feature + epsilon^2 effect size: eps2 = (H - k + 1) / (n - k)
kw = {}
for c in SEL:
    groups = [f.loc[lab.index[lab == g], c] for g in sorted(lab.unique())]
    H, p = stats.kruskal(*groups); n, k = len(lab), lab.nunique()
    kw[c] = {"H": float(H), "p": float(p), "epsilon2": float((H - k + 1) / (n - k))}
P["kruskal"] = kw
# categorical associations
cat = st.loc[labels.index]
plans = pd.read_csv(os.path.join(DATA_DIR, "_ground_truth", "SUBSCRIPTION_PLANS.csv"))
days = plans.groupby("STUDENTID").DAYS_PER_WEEK.max()
cat["PLAN_DAYS"] = cat.index.map(days).fillna(0).astype(int).astype(str)   # 0 = never subscribed (plan days are observable via STUDENTTYPE codes)
for col in ["BAND", "SCHOOLTYPE", "PLAN_DAYS"]:
    ct = pd.crosstab(lab, cat[col]); chi2, p, dof, v = cramers_v(ct)
    P[f"assoc_{col}"] = {"chi2": chi2, "p": p, "dof": dof, "cramers_v": v,
                         "dist": pd.crosstab(lab, cat[col], normalize="index").round(3).to_dict("index")}
# cuisine mix per cluster
tr = train[train.STUDENTID.isin(labels.index)].copy(); tr["CL"] = tr.STUDENTID.map(lab)
P["cuisine_mix"] = pd.crosstab(tr.CL, tr.CUISINENAME, normalize="index").round(3).to_dict("index")
P["top_items"] = {int(c): tr[tr.CL == c].ITEMNAME.value_counts(normalize=True).head(5).round(3).to_dict() for c in sorted(lab.unique())}
R["profile"] = P; save_R()
# figures
fig, ax = plt.subplots(figsize=(8, 3.2)); im = ax.imshow(zprof.values, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
ax.set_yticks(range(len(zprof)), [f"Cluster {i}" for i in zprof.index]); ax.set_xticks(range(len(SEL)), SEL, rotation=40, ha="right", fontsize=7)
for i in range(zprof.shape[0]):
    for j in range(zprof.shape[1]): ax.text(j, i, f"{zprof.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.grid(False); ax.set_title("Cluster profiles: mean z-score of each engineered feature"); plt.colorbar(im); savefig("p01_profile_heatmap")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
pd.crosstab(lab, cat.BAND, normalize="index")[BANDS].plot.bar(stacked=True, ax=ax[0], colormap="Blues", title="Age band mix by cluster")
pd.crosstab(lab, cat.PLAN_DAYS, normalize="index").plot.bar(stacked=True, ax=ax[1], colormap="Oranges", title="Subscription plan days by cluster (0 = none)")
for a in ax: a.legend(fontsize=6, bbox_to_anchor=(1, 1)); a.set_xlabel("cluster")
savefig("p02_cluster_composition")
print(prof.round(3).T.to_string()); print(zprof.round(2).T.to_string())
print({c: round(v["epsilon2"], 3) for c, v in kw.items()})
for col in ["BAND", "SCHOOLTYPE", "PLAN_DAYS"]: print(col, round(P[f"assoc_{col}"]["cramers_v"], 3), P[f"assoc_{col}"]["dist"])
print(P["top_items"])

## 8. Clusters → research questions
* **RQ1:** baseline Hit@1 by cluster.
* **RQ2:** the post-menu-change drop by cluster.
* **RQ3:** how many orders are needed to place a student in the right cluster.
* **RQ4:** auto-order cancellation by cluster.

In [ ]:
# ---------------- 8. CLUSTERS -> RESEARCH QUESTIONS ----------------
from sklearn.preprocessing import PowerTransformer, StandardScaler
labels = pd.read_pickle(os.path.join(CACHE, "labels.pkl")); lab = labels.R2
NAMES = {0: "Adventurous explorers", 1: "Committed subscribers", 2: "Comfort-food loyalists"}
CQ = {"names": NAMES}
stu_alg2 = stu_alg[stu_alg.ACTIVECODE.astype(str).str.lower() == "true"]
item_alg = pd.read_csv(os.path.join(DATA_DIR, "MENUITEMALLERGEN.csv"), encoding="utf-8-sig")
conflict = stu_alg2[["STUDENTID", "ALLERGENID"]].merge(item_alg[["MENUITEMID", "ALLERGENID"]], on="ALLERGENID")[["STUDENTID", "MENUITEMID"]].drop_duplicates()
conflict["X"] = 1
cand = menu[menu.PORTIONCODE == 1][["PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates()
def pick(C, cols): return C.sort_values(cols, ascending=False).drop_duplicates("EVENT").set_index("EVENT").MENUITEMID

# ---- RQ1: baseline Hit@1 by cluster (test window) ----
ev = test[test.STUDENTID.isin(lab.index)][["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "SCHOOLID", "BAND", "DELIVERYDATE", "MENUITEMID"]]
ev = ev.rename(columns={"STUDENTLUNCHID": "EVENT", "MENUITEMID": "CHOSEN"})
ev = ev.astype({"SCHOOLID": "int64", "PACKINGGROUPID": "int64"})
C = ev.merge(cand, on=["PACKINGGROUPID", "DELIVERYDATE"]).merge(conflict, on=["STUDENTID", "MENUITEMID"], how="left")
C = C[C.X.isna()].drop(columns="X")
pop = train.groupby(["SCHOOLID", "BAND", "MENUITEMID"], observed=True).size().rename("pop").reset_index().astype({"SCHOOLID": "int64"})
glob = train.MENUITEMID.value_counts().rename("glob")
hist = train.groupby(["STUDENTID", "MENUITEMID"]).size().rename("hist").reset_index()
C = C.merge(pop, on=["SCHOOLID", "BAND", "MENUITEMID"], how="left").merge(hist, on=["STUDENTID", "MENUITEMID"], how="left")
C["glob"] = C.MENUITEMID.map(glob).fillna(0); C[["pop", "hist"]] = C[["pop", "hist"]].fillna(0)
E = ev.set_index("EVENT"); E["n_cand"] = C.groupby("EVENT").size(); E = E.dropna(subset=["n_cand"])
E["hit_pop"] = (pick(C, ["pop", "glob"]).reindex(E.index) == E.CHOSEN).astype(int)
E["hit_fav"] = (pick(C, ["hist", "pop", "glob"]).reindex(E.index) == E.CHOSEN).astype(int)
E["CL"] = E.STUDENTID.map(lab)
t1 = E.groupby("CL").agg(events=("hit_fav", "size"), hit_fav=("hit_fav", "mean"), hit_pop=("hit_pop", "mean"), n_cand=("n_cand", "mean"))
t1["gap"] = t1.hit_fav - t1.hit_pop
chi2, p, dof, v = cramers_v(pd.crosstab(E.CL, E.hit_fav))
CQ["rq1"] = {"table": t1.round(4).to_dict("index"), "chi2": chi2, "p": p, "cramers_v": v}

# ---- RQ2: post-change drop by cluster ----
m1 = menu[menu.PORTIONCODE == 1][["SCHOOLID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates()
m1["WEEK"] = m1.DELIVERYDATE.dt.to_period("W").dt.start_time
m1["SLOT"] = m1.MENUITEMID.astype(str) + "_" + m1.DELIVERYDATE.dt.dayofweek.astype(str)
sl = m1.groupby(["SCHOOLID", "WEEK"]).SLOT.apply(set).reset_index().sort_values(["SCHOOLID", "WEEK"])
sl["P1"] = sl.groupby("SCHOOLID").SLOT.shift(1); sl["P2"] = sl.groupby("SCHOOLID").SLOT.shift(2); sl["N"] = sl.SLOT.apply(len)
sl["CH"] = [np.nan if not (isinstance(b, set) and isinstance(c, set)) else len(a - (b | c)) / len(a) for a, b, c in zip(sl.SLOT, sl.P1, sl.P2)]
sl = sl[sl.N >= 15]; thr = max(.15, sl.CH.quantile(.9)); chg = sl[sl.CH > thr][["SCHOOLID", "WEEK"]]
sd = m1[["SCHOOLID", "DELIVERYDATE"]].drop_duplicates().sort_values(["SCHOOLID", "DELIVERYDATE"]); sd["DAYIDX"] = sd.groupby("SCHOOLID").cumcount()
cs = chg.merge(sd, on="SCHOOLID"); cs = cs[cs.DELIVERYDATE >= cs.WEEK].sort_values("DELIVERYDATE").drop_duplicates(["SCHOOLID", "WEEK"])
cs = cs.astype({"SCHOOLID": "int64"})[["SCHOOLID", "DELIVERYDATE", "DAYIDX"]].rename(columns={"DELIVERYDATE": "CHG_DATE", "DAYIDX": "CHG_IDX"})
E2 = E.reset_index().merge(sd.astype({"SCHOOLID": "int64"}), on=["SCHOOLID", "DELIVERYDATE"], how="left").sort_values("DELIVERYDATE")
E2 = pd.merge_asof(E2, cs.sort_values("CHG_DATE"), left_on="DELIVERYDATE", right_on="CHG_DATE", by="SCHOOLID", direction="backward")
E2["POST"] = (E2.DAYIDX - E2.CHG_IDX).between(0, 9)
rq2 = {}
for c in sorted(lab.unique()):
    x = E2[E2.CL == c]; a, b = x[~x.POST].hit_fav, x[x.POST].hit_fav
    p1, p2 = a.mean(), b.mean(); pp = (a.sum() + b.sum()) / (len(a) + len(b))
    z = (p1 - p2) / math.sqrt(pp * (1 - pp) * (1 / len(a) + 1 / len(b)))
    rq2[int(c)] = {"mid": p1, "post": p2, "drop_pts": (p1 - p2) * 100, "z": z, "p": 2 * stats.norm.sf(abs(z)), "n_post": int(len(b))}
CQ["rq2"] = rq2

# ---- RQ3: how early can a student be placed in their cluster? ----
f = pd.read_pickle(os.path.join(CACHE, "f_raw.pkl")); SEL = R["features"]["selected"]; SK = R["features"]["yeo_johnson"]
pt = PowerTransformer(method="yeo-johnson").fit(f[SK]); tmp = f[SEL].copy(); tmp[SK] = pt.transform(tmp[SK]); sc = StandardScaler().fit(tmp)
km2 = pd.read_pickle(os.path.join(CACHE, "km2.pkl"))
COMFORT = {"Classic American", "Italian", "Breakfast for Lunch"}
def feats(t, placed):
    g = t.groupby("STUDENTID"); o = pd.DataFrame(index=g.size().index)
    o["orders_per_week"] = g.size() / ((g.DELIVERYDATE.max() - g.DELIVERYDATE.min()).dt.days / 7).clip(lower=1)
    cnt = t.groupby(["STUDENTID", "MENUITEMID"]).size()
    o["variety_ratio"] = cnt.groupby(level=0).size() / g.size(); o["fav_share"] = cnt.groupby(level=0).max() / g.size()
    o["adv_mean"] = g.ADVENTUROUSRATING.mean(); o["comfort_share"] = t.CUISINENAME.isin(COMFORT).groupby(t.STUDENTID).mean()
    cc = t.groupby(["STUDENTID", "CUISINENAME"]).size(); pc = cc / cc.groupby(level=0).transform("sum")
    o["cuisine_entropy"] = -(pc * np.log(pc)).groupby(level=0).sum() / np.log(10)
    o["sides_mean"] = g.SIDES.mean(); wd = t.groupby(["STUDENTID", "WD"]).size(); o["weekday_conc"] = wd.groupby(level=0).max() / g.size()
    o["sub_share"] = g.IS_SUB.mean(); o["cold_share"] = (t.SERVINGTYPE == "Cold").groupby(t.STUDENTID).mean()
    o["cancel_rate"] = placed.groupby("STUDENTID").CANCELLED.mean().reindex(o.index).fillna(0)
    return o[SEL]
tr = train[train.STUDENTID.isin(lab.index)].sort_values(["STUDENTID", "DELIVERYDATE"]).copy()
tr["K"] = tr.groupby("STUDENTID").cumcount() + 1
pl = lunch[(lunch.DELIVERYDATE <= TRAIN_END) & (lunch.LUNCHTYPE != "IOU") & lunch.STUDENTID.isin(lab.index)]
rq3 = {}
for k in [3, 5, 10, 20, 40]:
    tk = tr[tr.K <= k]; ok = tk.groupby("STUDENTID").size() == k; ids = ok[ok].index
    cut = tk[tk.STUDENTID.isin(ids)].groupby("STUDENTID").DELIVERYDATE.max()
    plk = pl[pl.STUDENTID.isin(ids)]; plk = plk[plk.DELIVERYDATE <= plk.STUDENTID.map(cut)]
    o = feats(tk[tk.STUDENTID.isin(ids)], plk); o[SK] = pt.transform(o[SK])
    pred = km2.predict(sc.transform(o[SEL]))
    acc = float((pred == lab.loc[o.index].values).mean())
    per = pd.Series(pred == lab.loc[o.index].values, index=o.index).groupby(lab.loc[o.index]).mean().round(4).to_dict()
    rq3[k] = {"accuracy": acc, "n": int(len(o)), "per_cluster": per}
CQ["rq3"] = rq3

# ---- RQ4: auto-order cancellation by cluster ----
au = lunch[(lunch.LUNCHTYPE == "AutoOrder") & lunch.STUDENTID.isin(lab.index)].copy(); au["CL"] = au.STUDENTID.map(lab)
t4 = au.groupby("CL").agg(auto_orders=("CANCELLED", "size"), cancel_rate=("CANCELLED", "mean"))
pre = lunch[(lunch.LUNCHTYPE == "PreOrder") & lunch.STUDENTID.isin(lab.index)].copy(); pre["CL"] = pre.STUDENTID.map(lab)
t4["preorder_cancel"] = pre.groupby("CL").CANCELLED.mean()
t4["students_with_auto"] = au.groupby("CL").STUDENTID.nunique()
chi2, p, dof, v = cramers_v(pd.crosstab(au.CL, au.CANCELLED))
CQ["rq4"] = {"table": t4.round(4).to_dict("index"), "chi2": chi2, "p": p, "cramers_v": v}
R["cluster_rq"] = CQ; save_R()

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2)); xl = [NAMES[i].split()[0] + "\n" + NAMES[i].split()[1] for i in range(3)]
x = np.arange(3); ax[0].bar(x - .2, t1.hit_fav, .4, color=NAVY, label="repeat-favourite"); ax[0].bar(x + .2, t1.hit_pop, .4, color=ORANGE, label="popularity")
ax[0].set_xticks(x, xl, fontsize=7); ax[0].set_title("RQ1: baseline Hit@1 by cluster"); ax[0].legend(fontsize=6)
ax[1].bar(x, [rq2[i]["drop_pts"] for i in range(3)], color=NAVY); ax[1].set_xticks(x, xl, fontsize=7); ax[1].set_title("RQ2: Hit@1 drop after menu change (pts)")
ks = list(rq3); ax[2].plot(ks, [rq3[k]["accuracy"] for k in ks], marker="o", color=NAVY); ax[2].set_xlabel("orders observed")
ax[2].set_title("RQ3: correct cluster from first k orders"); ax[2].set_ylim(0, 1)
ax[3].bar(x - .2, t4.cancel_rate, .4, color=NAVY, label="auto-order"); ax[3].bar(x + .2, t4.preorder_cancel, .4, color=ORANGE, label="pre-order")
ax[3].set_xticks(x, xl, fontsize=7); ax[3].set_title("RQ4: cancellation rate by cluster"); ax[3].legend(fontsize=6)
savefig("q01_clusters_vs_rqs")
print(t1.round(3)); print({k: {kk: round(vv, 4) if isinstance(vv, float) else vv for kk, vv in v_.items()} for k, v_ in rq2.items()})
print({k: (round(v_["accuracy"], 3), v_["n"]) for k, v_ in rq3.items()}); print(t4.round(3)); print("rq1 V", round(CQ["rq1"]["cramers_v"], 3), "rq4 V", round(CQ["rq4"]["cramers_v"], 3), CQ["rq4"]["p"])

## 9. Data cleaning audit
Counts for the cleaning log: missing values (true blanks and export placeholders), duplicates (exact rows, keys and logical duplicates), and outliers using the interquartile-range rule (outside Q1 − 1.5·IQR or Q3 + 1.5·IQR).

In [ ]:
# ---------------- 9. DATA CLEANING AUDIT (counts for the cleaning log) ----------------
CL = {}
# stream the 354 MB lunch file in chunks (keeps memory low): count blanks/placeholders, hash rows for duplicates
miss_b, miss_a, hashes, ids, live_keys, N = {}, {}, [], [], [], 0
isserved, cancelled, iou = {}, 0, 0
for ch in pd.read_csv(os.path.join(DATA_DIR, "STUDENTLUNCH.csv"), encoding="utf-8-sig", dtype=str, keep_default_na=False, chunksize=400000):
    N += len(ch)
    for c in ch.columns:
        miss_b[c] = miss_b.get(c, 0) + int((ch[c] == "").sum()); miss_a[c] = miss_a.get(c, 0) + int((ch[c] == '""').sum())
    hashes.append(pd.util.hash_pandas_object(ch, index=False).values); ids.append(ch.STUDENTLUNCHID.values)
    lv = ch[ch.LUNCHCANCELED == ""]; live_keys.append(pd.util.hash_pandas_object(lv[["STUDENTID", "DELIVERYDATE"]], index=False).values)
    for k, v in ch.ISSERVED.value_counts().items(): isserved[k] = isserved.get(k, 0) + int(v)
    cancelled += int((ch.LUNCHCANCELED != "").sum()); iou += int((ch.LUNCHTYPE == "IOU").sum()); cols = list(ch.columns)
CL["lunch_rows"] = N; CL["lunch_cols"] = len(cols)
CL["missing_lunch"] = {c: {"blank": miss_b[c], "artifact": miss_a[c], "pct": (miss_b[c] + miss_a[c]) / N} for c in cols if miss_b[c] + miss_a[c] > 0}
CL["dup_exact_rows"] = int(pd.Series(np.concatenate(hashes)).duplicated().sum())
CL["dup_lunch_id"] = int(pd.Series(np.concatenate(ids)).duplicated().sum())
CL["dup_live_student_day"] = int(pd.Series(np.concatenate(live_keys)).duplicated().sum())
CL["isserved_values"] = isserved; CL["cancelled"] = cancelled; CL["iou"] = iou
del hashes, ids, live_keys
stu = pd.read_csv(os.path.join(DATA_DIR, "STUDENT.csv"), encoding="utf-8-sig", dtype=str, keep_default_na=False)
CL["student_rows"], CL["student_cols"] = stu.shape
CL["missing_student"] = {c: float(((stu[c] == "") | (stu[c] == '""')).mean()) for c in stu.columns if ((stu[c] == "") | (stu[c] == '""')).any()}
CL["dup_student_id"] = int(stu.STUDENTID.duplicated().sum())
mn = menu; CL["dup_menu_rows"] = int(mn.duplicated(["PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID", "PORTIONCODE"]).sum())
ad = pd.read_csv(os.path.join(DATA_DIR, "STUDENTLUNCHADDON.csv"), encoding="utf-8-sig", usecols=["ADDONID", "STUDENTLUNCHID", "COMPONENTID"])
CL["addon_rows"] = len(ad); CL["dup_addon_id"] = int(ad.ADDONID.duplicated().sum())
CL["dup_addon_same_component"] = int(ad.duplicated(["STUDENTLUNCHID", "COMPONENTID"]).sum())
del ad
# outliers: IQR rule  x < Q1 - 1.5 IQR  or  x > Q3 + 1.5 IQR
def iqr_out(s):
    s = pd.Series(s).dropna(); q1, q3 = s.quantile(.25), s.quantile(.75); i = q3 - q1
    lo, hi = q1 - 1.5 * i, q3 + 1.5 * i
    return {"n": int(len(s)), "low_fence": float(lo), "high_fence": float(hi), "n_out": int(((s < lo) | (s > hi)).sum()),
            "pct_out": float(((s < lo) | (s > hi)).mean()), "max": float(s.max()), "min": float(s.min())}
lead = lunch.loc[lunch.LUNCHTYPE.isin(["PreOrder", "LastMinute"]), "LEAD_H"]
CL["out_lead_time"] = iqr_out(lead)
CL["out_daily_volume"] = iqr_out(live.groupby("DELIVERYDATE").size())
st = pd.read_pickle(os.path.join(CACHE, "student_raw.pkl"))
CL["out_lunches_per_student"] = iqr_out(st.loc[st.N_LUNCH > 0, "N_LUNCH"])
f = pd.read_pickle(os.path.join(CACHE, "f_raw.pkl"))
CL["out_features"] = {c: iqr_out(f[c]) for c in R["features"]["selected"]}
CL["never_ordered"] = int((st.N_LUNCH == 0).sum())
CL["below_min_orders"] = int(((train.groupby("STUDENTID").size() < MIN_ORDERS)).sum())
R["cleaning"] = CL; save_R()
print(json.dumps({k: v for k, v in CL.items() if k not in ("out_features",)}, indent=0, default=float)[:3500])
print({c: (v["n_out"], round(v["pct_out"], 4)) for c, v in CL["out_features"].items()})

import gc; gc.collect()

## 10. Preliminary models: RQ1 (ranking the day's eligible entrées)
* **Sample:** 3,000 students, stratified by segment.
* **Training events:** January–March 2025, with features computed only from history up to 31 December 2024.
* **Test events:** April–June 2025, with features computed from history up to 31 March 2025.
* **Models compared:** popularity and repeat-favourite baselines, logistic regression, and LightGBM LambdaRank with and without segment features.
* **Significance:** McNemar's test on events, and a paired t test on per-student Hit@1.

In [ ]:
# ---------------- 10. PRELIMINARY MODELS - RQ1 (rank the day's eligible entrees) ----------------
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
labels = pd.read_pickle(os.path.join(CACHE, "labels.pkl")); lab = labels.R2
N_SAMPLE = 3000
samp = lab.groupby(lab, group_keys=False).apply(lambda s: s.sample(int(round(N_SAMPLE * len(s) / len(lab))), random_state=SEED)).index
item_alg = pd.read_csv(os.path.join(DATA_DIR, "MENUITEMALLERGEN.csv"), encoding="utf-8-sig")
act_alg = stu_alg[stu_alg.ACTIVECODE.astype(str).str.lower() == "true"]
conflict = act_alg[["STUDENTID", "ALLERGENID"]].merge(item_alg[["MENUITEMID", "ALLERGENID"]], on="ALLERGENID")[["STUDENTID", "MENUITEMID"]].drop_duplicates()
conflict["X"] = 1
cand = menu[menu.PORTIONCODE == 1][["PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates().astype({"PACKINGGROUPID": "int64"})
it = item.set_index("MENUITEMID")
COMFORT = {"Classic American", "Italian", "Breakfast for Lunch"}
itf = pd.DataFrame({"item_adv": it.ADVENTUROUSRATING, "item_cold": (it.SERVINGTYPE == "Cold").astype(int),
                    "item_comfort": it.CUISINENAME.isin(COMFORT).astype(int), "CUISINENAME": it.CUISINENAME})
CL_DUMMIES = pd.get_dummies(lab.loc[samp], prefix="cl").astype(int)

def build(ev_start, ev_end, hist_end):
    """candidate rows for events in [ev_start, ev_end] with features from history <= hist_end (no leakage)"""
    L = live[live.STUDENTID.isin(samp)]
    H = L[L.DELIVERYDATE <= hist_end]; E = L[(L.DELIVERYDATE >= ev_start) & (L.DELIVERYDATE <= ev_end)]
    ev = E[["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "SCHOOLID", "BAND", "DELIVERYDATE", "MENUITEMID", "WD"]].rename(
        columns={"STUDENTLUNCHID": "EVENT", "MENUITEMID": "CHOSEN"}).astype({"PACKINGGROUPID": "int64", "SCHOOLID": "int64"})
    C = ev.merge(cand, on=["PACKINGGROUPID", "DELIVERYDATE"]).merge(conflict, on=["STUDENTID", "MENUITEMID"], how="left")
    C = C[C.X.isna()].drop(columns="X")
    n_h = H.groupby("STUDENTID").size()
    cnt = H.groupby(["STUDENTID", "MENUITEMID"]).size().rename("hist_cnt").reset_index()
    dcnt = H.groupby(["STUDENTID", "WD", "MENUITEMID"]).size().rename("dow_cnt").reset_index()
    dtot = H.groupby(["STUDENTID", "WD"]).size().rename("dow_tot").reset_index()
    last = H.groupby(["STUDENTID", "MENUITEMID"]).DELIVERYDATE.max().rename("last").reset_index()
    pop = live[live.DELIVERYDATE <= hist_end].groupby(["SCHOOLID", "BAND", "MENUITEMID"], observed=True).size().rename("pop_cnt").reset_index().astype({"SCHOOLID": "int64"})
    popt = pop.groupby(["SCHOOLID", "BAND"], observed=True).pop_cnt.sum().rename("pop_tot").reset_index()
    cui = H.groupby(["STUDENTID", "CUISINENAME"], observed=True).size().rename("cui_cnt").reset_index()
    sadv = H.groupby("STUDENTID").ADVENTUROUSRATING.mean().rename("stu_adv")
    scnt = H.groupby(["STUDENTID", "MENUITEMID"]).size()
    sfav = (scnt.groupby(level=0).max() / n_h).rename("stu_fav"); svar = (scnt.groupby(level=0).size() / n_h).rename("stu_var")
    C = (C.merge(cnt, on=["STUDENTID", "MENUITEMID"], how="left").merge(dcnt, on=["STUDENTID", "WD", "MENUITEMID"], how="left")
          .merge(dtot, on=["STUDENTID", "WD"], how="left").merge(last, on=["STUDENTID", "MENUITEMID"], how="left")
          .merge(pop, on=["SCHOOLID", "BAND", "MENUITEMID"], how="left").merge(popt, on=["SCHOOLID", "BAND"], how="left")
          .join(itf, on="MENUITEMID"))
    C["CUISINENAME"] = C.CUISINENAME.astype(str)
    C = C.merge(cui.astype({"CUISINENAME": str}), on=["STUDENTID", "CUISINENAME"], how="left")
    C = C.join(sadv, on="STUDENTID").join(sfav, on="STUDENTID").join(svar, on="STUDENTID").join(n_h.rename("n_hist"), on="STUDENTID")
    C = C.join(CL_DUMMIES, on="STUDENTID")
    C[["hist_cnt", "dow_cnt", "pop_cnt", "cui_cnt"]] = C[["hist_cnt", "dow_cnt", "pop_cnt", "cui_cnt"]].fillna(0)
    C["hist_share"] = C.hist_cnt / C.n_hist; C["dow_share"] = C.dow_cnt / C.dow_tot.fillna(0).clip(lower=1)
    C["days_since"] = (C.DELIVERYDATE - C["last"]).dt.days.fillna(400).clip(upper=400)
    C["pop_share"] = C.pop_cnt / C.pop_tot.clip(lower=1); C["cui_share"] = C.cui_cnt / C.n_hist
    C["adv_gap"] = (C.item_adv - C.stu_adv).abs(); C["n_cand"] = C.groupby("EVENT").MENUITEMID.transform("size")
    C["label"] = (C.MENUITEMID == C.CHOSEN).astype(int)
    C = C[C.groupby("EVENT").label.transform("sum") == 1].sort_values(["EVENT", "MENUITEMID"]).reset_index(drop=True)
    return C
BASE_F = ["hist_share", "dow_share", "days_since", "pop_share", "cui_share", "item_adv", "adv_gap", "item_cold",
          "item_comfort", "n_cand", "stu_fav", "stu_var"]
CLF = list(CL_DUMMIES.columns); ALL_F = BASE_F + CLF
TR = cache("rq1_train", lambda: build("2025-01-01", TRAIN_END, "2024-12-31"))
TE = cache("rq1_test", lambda: build(TEST_START, "2025-06-30", TRAIN_END))
def fill(C):
    # students with no earlier history (cold start): history shares = 0, student traits = training medians
    C = C.copy(); C["n_hist"] = C.n_hist.fillna(0)
    for c in ["hist_share", "dow_share", "cui_share"]: C[c] = C[c].fillna(0)
    for c in ["stu_adv", "stu_fav", "stu_var"]: C[c] = C[c].fillna(TR_MED[c])
    C["adv_gap"] = (C.item_adv - C.stu_adv).abs()
    for c in CLF: C[c] = C[c].fillna(0)
    return C
TR_MED = TR[["stu_adv", "stu_fav", "stu_var"]].median()
R_NAN = {"train_rows_with_nan": int(TR[BASE_F].isna().any(axis=1).sum()), "test_rows_with_nan": int(TE[BASE_F].isna().any(axis=1).sum())}
TR, TE = fill(TR), fill(TE)
print("NaN rows filled:", R_NAN, flush=True)
print("train rows", len(TR), "events", TR.EVENT.nunique(), "| test rows", len(TE), "events", TE.EVENT.nunique(), flush=True)

def hit_ndcg(df, score):
    d = df.assign(s=score).sort_values(["EVENT", "s"], ascending=[True, False])
    d["rank"] = d.groupby("EVENT").cumcount() + 1
    r = d[d.label == 1].set_index("EVENT")
    hit = (r["rank"] == 1).astype(int); nd = np.where(r["rank"] <= 3, 1 / np.log2(r["rank"] + 1), 0.0)
    return hit, pd.Series(nd, index=r.index)
res, hits = {}, {}
rng = np.random.default_rng(SEED)
for name, sc in [("Popularity (baseline)", TE.pop_share + 1e-6 * rng.random(len(TE))),
                 ("Repeat-favourite (baseline)", TE.hist_share + 1e-3 * TE.pop_share)]:
    h, n = hit_ndcg(TE, sc.values); res[name] = {"hit1": h.mean(), "ndcg3": n.mean()}; hits[name] = h
# logistic regression (pointwise): P(chosen | candidate features)
scl = StandardScaler().fit(TR[ALL_F]); lr = LogisticRegression(max_iter=1000, C=1.0).fit(scl.transform(TR[ALL_F]), TR.label)
h, n = hit_ndcg(TE, lr.predict_proba(scl.transform(TE[ALL_F]))[:, 1]); res["Logistic regression"] = {"hit1": h.mean(), "ndcg3": n.mean()}; hits["Logistic regression"] = h
LR_COEF = dict(zip(ALL_F, lr.coef_[0].round(4)))
# LightGBM LambdaRank, with and without cluster features; last 15% of training events (by date) for early stopping
ev_dates = TR.groupby("EVENT").DELIVERYDATE.first(); cut = ev_dates.quantile(.85)
va_ev = ev_dates[ev_dates > cut].index
def fit_rank(F):
    tr, va = TR[~TR.EVENT.isin(va_ev)], TR[TR.EVENT.isin(va_ev)]
    m = lgb.LGBMRanker(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    m.fit(tr[F], tr.label, group=tr.groupby("EVENT", sort=False).size().values, eval_set=[(va[F], va.label)],
          eval_group=[va.groupby("EVENT", sort=False).size().values], eval_at=[1], callbacks=[lgb.early_stopping(30, verbose=False)])
    return m
for name, F in [("LightGBM ranker (no cluster features)", BASE_F), ("LightGBM ranker (with cluster features)", ALL_F)]:
    m = fit_rank(F); h, n = hit_ndcg(TE, m.predict(TE[F])); res[name] = {"hit1": h.mean(), "ndcg3": n.mean(), "trees": m.best_iteration_}
    hits[name] = h
    if "with" in name:
        imp = pd.Series(m.booster_.feature_importance("gain"), index=F); IMP = (imp / imp.sum()).sort_values(ascending=False)
# significance: best model vs repeat-favourite (McNemar on events; paired t on per-student Hit@1)
A, Bh = hits["LightGBM ranker (with cluster features)"], hits["Repeat-favourite (baseline)"]
b = int(((A == 1) & (Bh == 0)).sum()); c = int(((A == 0) & (Bh == 1)).sum())
mc = (abs(b - c) - 1) ** 2 / (b + c)
stu_of = TE.drop_duplicates("EVENT").set_index("EVENT").STUDENTID
ps = pd.DataFrame({"m": A, "b": Bh}).join(stu_of).groupby("STUDENTID").mean()
t, p_t = stats.ttest_rel(ps.m, ps.b); d = ps.m - ps.b
# per-cluster Hit@1
clh = pd.DataFrame({k: v for k, v in hits.items()}).join(stu_of).assign(CL=lambda x: x.STUDENTID.map(lab)).groupby("CL").mean(numeric_only=True)
R["rq1_models"] = {"nan_filled": R_NAN, "results": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in res.items()},
                   "n_train_rows": int(len(TR)), "n_train_events": int(TR.EVENT.nunique()), "n_test_rows": int(len(TE)),
                   "n_test_events": int(TE.EVENT.nunique()), "n_students": int(len(samp)), "test_students": int(ps.shape[0]),
                   "mcnemar": {"b": b, "c": c, "chi2": float(mc), "p": float(stats.chi2.sf(mc, 1))},
                   "paired_t": {"t": float(t), "p": float(p_t), "dz": float(d.mean() / d.std()), "mean_diff": float(d.mean())},
                   "importance": IMP.round(4).to_dict(), "lr_coef": {k: float(v) for k, v in LR_COEF.items()},
                   "by_cluster": clh.round(4).to_dict("index")}
save_R()
pd.to_pickle(hits, os.path.join(CACHE, "rq1_hits.pkl"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
rs = pd.DataFrame(res).T
ax[0].barh(rs.index[::-1], rs.hit1[::-1], color=[NAVY if "LightGBM" in i else ("#7f8fa6" if "Logistic" in i else ORANGE) for i in rs.index[::-1]])
for i, v in enumerate(rs.hit1[::-1]): ax[0].text(v + .005, i, f"{v:.3f}", va="center", fontsize=8)
ax[0].set_xlabel("Hit@1 on test window (Apr–Jun 2025)"); ax[0].set_title("RQ1: model comparison"); ax[0].set_xlim(0, max(rs.hit1) * 1.15)
ax[1].barh(IMP.index[::-1], IMP.values[::-1], color=NAVY); ax[1].set_title("LightGBM ranker: feature importance (share of gain)")
savefig("m01_rq1_models")
print(pd.DataFrame(res).T.round(4)); print(R["rq1_models"]["mcnemar"], R["rq1_models"]["paired_t"]); print(IMP.round(3).head(8).to_dict()); print(clh.round(3))

import gc; del TR, TE; gc.collect()

## 11. Preliminary models: RQ2 (menu-change phase) and RQ4 (cancellation classifier)
* **RQ2:** Hit@1 of each model, mid-cycle versus the ten school days after a detected menu change.
* **RQ4:** base-rate and mismatch-rule baselines, logistic regression, and a LightGBM classifier, trained on auto-orders up to 31 March 2025 and tested on April–June 2025. The decision threshold is chosen on training data only, and a student-clustered bootstrap gives an interval for the AUC difference.

In [ ]:
# ---------------- 11. PRELIMINARY MODELS - RQ2 (phase) and RQ4 (cancellation classifier) ----------------
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_fscore_support, accuracy_score
labels = pd.read_pickle(os.path.join(CACHE, "labels.pkl")); lab = labels.R2
hits = pd.read_pickle(os.path.join(CACHE, "rq1_hits.pkl")); TE = pd.read_pickle(os.path.join(CACHE, "rq1_test.pkl"))
# ---- RQ2: menu-cycle phase of each test event ----
m1 = menu[menu.PORTIONCODE == 1][["SCHOOLID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates()
m1["WEEK"] = m1.DELIVERYDATE.dt.to_period("W").dt.start_time
m1["SLOT"] = m1.MENUITEMID.astype(str) + "_" + m1.DELIVERYDATE.dt.dayofweek.astype(str)
sl = m1.groupby(["SCHOOLID", "WEEK"]).SLOT.apply(set).reset_index().sort_values(["SCHOOLID", "WEEK"])
sl["P1"] = sl.groupby("SCHOOLID").SLOT.shift(1); sl["P2"] = sl.groupby("SCHOOLID").SLOT.shift(2); sl["N"] = sl.SLOT.apply(len)
sl["CH"] = [np.nan if not (isinstance(b, set) and isinstance(c, set)) else len(a - (b | c)) / len(a) for a, b, c in zip(sl.SLOT, sl.P1, sl.P2)]
sl = sl[sl.N >= 15]; thr = max(.15, sl.CH.quantile(.9)); chg = sl[sl.CH > thr][["SCHOOLID", "WEEK"]]
sd = m1[["SCHOOLID", "DELIVERYDATE"]].drop_duplicates().sort_values(["SCHOOLID", "DELIVERYDATE"]); sd["DAYIDX"] = sd.groupby("SCHOOLID").cumcount()
cs = chg.merge(sd, on="SCHOOLID"); cs = cs[cs.DELIVERYDATE >= cs.WEEK].sort_values("DELIVERYDATE").drop_duplicates(["SCHOOLID", "WEEK"])
cs = cs[["SCHOOLID", "DELIVERYDATE", "DAYIDX"]].rename(columns={"DELIVERYDATE": "CHG_DATE", "DAYIDX": "CHG_IDX"}).astype({"SCHOOLID": "int64"})
ev = TE.drop_duplicates("EVENT")[["EVENT", "SCHOOLID", "DELIVERYDATE"]].astype({"SCHOOLID": "int64"})
ev = ev.merge(sd.astype({"SCHOOLID": "int64"}), on=["SCHOOLID", "DELIVERYDATE"], how="left").sort_values("DELIVERYDATE")
ev = pd.merge_asof(ev, cs.sort_values("CHG_DATE"), left_on="DELIVERYDATE", right_on="CHG_DATE", by="SCHOOLID", direction="backward")
ev["POST"] = (ev.DAYIDX - ev.CHG_IDX).between(0, 9); post = ev.set_index("EVENT").POST
rq2 = {}
for k in ["Repeat-favourite (baseline)", "Logistic regression", "LightGBM ranker (with cluster features)"]:
    h = hits[k]; ph = post.reindex(h.index).fillna(False)
    a, b = h[~ph], h[ph]; p1, p2 = a.mean(), b.mean(); P = (a.sum() + b.sum()) / (len(a) + len(b))
    z = (p1 - p2) / math.sqrt(P * (1 - P) * (1 / len(a) + 1 / len(b)))
    rq2[k] = {"mid": float(p1), "post": float(p2), "drop_pts": float((p1 - p2) * 100), "z": float(z), "p": float(2 * stats.norm.sf(abs(z))), "n_post": int(len(b)), "n_mid": int(len(a))}
R["rq2_models"] = rq2; save_R(); print(pd.DataFrame(rq2).T.round(4), flush=True)

# ---- RQ4: predict cancellation of legacy auto-orders ----
au = lunch[lunch.LUNCHTYPE == "AutoOrder"][["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "SCHOOLID", "BAND", "DELIVERYDATE",
                                              "MENUITEMID", "STUDENTTYPE", "WD", "CANCELLED", "ADVENTUROUSRATING"]].copy()
au = au.astype({"PACKINGGROUPID": "int64", "SCHOOLID": "int64"})
cand = menu[menu.PORTIONCODE == 1][["PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates().astype({"PACKINGGROUPID": "int64"})
C = au[["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "DELIVERYDATE"]].merge(cand, on=["PACKINGGROUPID", "DELIVERYDATE"])
man = live[live.LUNCHTYPE.isin(["PreOrder", "LastMinute"]) & live.STUDENTID.isin(au.STUDENTID)][["STUDENTID", "MENUITEMID", "DELIVERYDATE"]].sort_values("DELIVERYDATE").copy()
man["PRIOR"] = man.groupby(["STUDENTID", "MENUITEMID"]).cumcount() + 1
C = pd.merge_asof(C.sort_values("DELIVERYDATE"), man, on="DELIVERYDATE", by=["STUDENTID", "MENUITEMID"], allow_exact_matches=False, direction="backward")
C["PRIOR"] = C.PRIOR.fillna(0)
tot = man.copy(); tot["N"] = tot.groupby("STUDENTID").cumcount() + 1
au = pd.merge_asof(au.sort_values("DELIVERYDATE"), tot[["STUDENTID", "DELIVERYDATE", "N"]].sort_values("DELIVERYDATE"), on="DELIVERYDATE",
                   by="STUDENTID", allow_exact_matches=False, direction="backward").rename(columns={"N": "n_manual"})
au["n_manual"] = au.n_manual.fillna(0)
best = C.sort_values(["STUDENTLUNCHID", "PRIOR"], ascending=[True, False]).drop_duplicates("STUDENTLUNCHID").set_index("STUDENTLUNCHID")
pick = C.merge(au[["STUDENTLUNCHID", "MENUITEMID"]], on=["STUDENTLUNCHID", "MENUITEMID"]).set_index("STUDENTLUNCHID").PRIOR
au = au.set_index("STUDENTLUNCHID")
au["pref_prior"] = best.PRIOR.reindex(au.index).fillna(0); au["pick_prior"] = pick.reindex(au.index).fillna(0)
au["pick_share"] = au.pick_prior / au.n_manual.clip(lower=1); au["pref_share"] = au.pref_prior / au.n_manual.clip(lower=1)
au["share_gap"] = au.pref_share - au.pick_share; au["match"] = ((au.pick_prior >= au.pref_prior) & (au.n_manual > 0)).astype(int)
au["n_cand"] = C.groupby("STUDENTLUNCHID").size().reindex(au.index).fillna(0)
codes = au.STUDENTTYPE.astype(str).str.extract(r"Sub (\d)(\d)").astype(float)
au["plan_days"], au["plan_sides"] = codes[0].fillna(0), codes[1].fillna(0)
pop = live[live.DELIVERYDATE <= TRAIN_END].groupby(["SCHOOLID", "BAND", "MENUITEMID"], observed=True).size().rename("pc").reset_index().astype({"SCHOOLID": "int64"})
pop["pick_pop"] = pop.pc / pop.groupby(["SCHOOLID", "BAND"], observed=True).pc.transform("sum")
au = au.reset_index().merge(pop[["SCHOOLID", "BAND", "MENUITEMID", "pick_pop"]], on=["SCHOOLID", "BAND", "MENUITEMID"], how="left").set_index("STUDENTLUNCHID")
au["pick_pop"] = au.pick_pop.fillna(0); au["log_n_manual"] = np.log1p(au.n_manual)
au["month"] = au.DELIVERYDATE.dt.month; au["pick_adv"] = au.ADVENTUROUSRATING
cl = au.STUDENTID.map(lab); au["cluster"] = cl.fillna(-1).astype(int)
au = au.join(pd.get_dummies(au.cluster, prefix="cl").astype(int)).join(pd.get_dummies(au.WD, prefix="wd").astype(int))
F4 = ["match", "pick_share", "pref_share", "share_gap", "log_n_manual", "pick_pop", "pick_adv", "n_cand", "plan_days", "plan_sides",
      "month"] + [c for c in au.columns if c.startswith("cl_") or c.startswith("wd_")]
tr, te = au[au.DELIVERYDATE <= TRAIN_END], au[au.DELIVERYDATE >= TEST_START]
y_tr, y_te = tr.CANCELLED.astype(int), te.CANCELLED.astype(int)
out = {}
def evaluate(name, p_tr, p_te):
    # threshold chosen on TRAINING data to maximise F1, then applied to test (no peeking)
    grid = np.linspace(.05, .6, 56); f1s = [precision_recall_fscore_support(y_tr, p_tr >= g, average="binary", zero_division=0)[2] for g in grid]
    th = float(grid[int(np.argmax(f1s))]); yhat = p_te >= th
    pr, rc, f1, _ = precision_recall_fscore_support(y_te, yhat, average="binary", zero_division=0)
    out[name] = {"auc": roc_auc_score(y_te, p_te), "pr_auc": average_precision_score(y_te, p_te), "brier": brier_score_loss(y_te, p_te),
                 "threshold": th, "precision": pr, "recall": rc, "f1": f1, "accuracy": accuracy_score(y_te, yhat)}
base = float(y_tr.mean()); evaluate("Base rate (baseline)", np.full(len(tr), base), np.full(len(te), base))
evaluate("Rule: mismatch flag (baseline)", 1 - tr.match.values * .5, 1 - te.match.values * .5)
sc4 = StandardScaler().fit(tr[F4]); lr4 = LogisticRegression(max_iter=2000).fit(sc4.transform(tr[F4]), y_tr)
p_lr = lr4.predict_proba(sc4.transform(te[F4]))[:, 1]; evaluate("Logistic regression", lr4.predict_proba(sc4.transform(tr[F4]))[:, 1], p_lr)
gb = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=100, subsample=0.8, subsample_freq=1,
                        colsample_bytree=0.8, random_state=SEED, verbose=-1)
cut = tr.DELIVERYDATE.quantile(.85); fit_, val_ = tr[tr.DELIVERYDATE <= cut], tr[tr.DELIVERYDATE > cut]
gb.fit(fit_[F4], fit_.CANCELLED.astype(int), eval_set=[(val_[F4], val_.CANCELLED.astype(int))], eval_metric="auc",
       callbacks=[lgb.early_stopping(50, verbose=False)])
p_gb = gb.predict_proba(te[F4])[:, 1]; evaluate("LightGBM classifier", gb.predict_proba(tr[F4])[:, 1], p_gb)
# bootstrap CI for AUC difference (LightGBM - logistic), resampling students (clusters of orders)
stu = te.STUDENTID.values; ust = np.unique(stu); rng = np.random.default_rng(SEED); idx_by = pd.Series(np.arange(len(te))).groupby(stu).apply(np.array)
diffs = []
for _ in range(200):
    pick_s = rng.choice(ust, len(ust)); ii = np.concatenate([idx_by[s] for s in pick_s])
    if y_te.values[ii].min() == y_te.values[ii].max(): continue
    diffs.append(roc_auc_score(y_te.values[ii], p_gb[ii]) - roc_auc_score(y_te.values[ii], p_lr[ii]))
imp4 = pd.Series(gb.booster_.feature_importance("gain"), index=F4); imp4 = (imp4 / imp4.sum()).sort_values(ascending=False)
R["rq4_models"] = {"results": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in out.items()},
                   "n_train": int(len(tr)), "n_test": int(len(te)), "pos_rate_train": float(y_tr.mean()), "pos_rate_test": float(y_te.mean()),
                   "auc_diff_ci": [float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))], "auc_diff_mean": float(np.mean(diffs)),
                   "importance": imp4.round(4).to_dict(), "lr_coef": dict(zip(F4, lr4.coef_[0].round(4).tolist())), "trees": int(gb.best_iteration_)}
save_R()
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for nm, p, c in [("Logistic regression", p_lr, "#7f8fa6"), ("LightGBM classifier", p_gb, NAVY)]:
    fpr, tpr, _ = roc_curve(y_te, p); ax[0].plot(fpr, tpr, color=c, label=f"{nm} (AUC {out[nm]['auc']:.3f})")
ax[0].plot([0, 1], [0, 1], ls="--", color="grey", lw=.8, label="Chance (AUC 0.500)"); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
ax[0].set_title("RQ4: ROC curves, test window"); ax[0].legend(fontsize=7)
top = imp4.head(10); ax[1].barh(top.index[::-1], top.values[::-1], color=NAVY); ax[1].set_title("LightGBM classifier: feature importance (share of gain)")
savefig("m02_rq4_models")
print(pd.DataFrame(out).T.round(4)); print("AUC diff CI", R["rq4_models"]["auc_diff_ci"]); print(imp4.head(8).round(3).to_dict()); print(len(tr), len(te), y_te.mean())

import gc; del C, au, tr, te, man, tot, best; gc.collect()